# NexTwin AI — Industrial Digital Twin Platform
## Notebook 05: Production Bottleneck Prediction Model

### Objectives
1. **Load Integrated Dataset**: Load `engineered_mfg_bottleneck.csv` containing blended utilization, production, and sensor logs.
2. **Predict Bottleneck Indicators**:
   - **Bottleneck Risk Score**: Predict the `bottleneck_severity_index` (Regression).
   - **Production Delay**: Predict `production_delay` (Regression, computed as `target_quantity - actual_quantity`).
   - **Congestion Risk**: Predict high congestion states (Classification, `utilization_rate > 90%`).
3. **Train Models**: Train Random Forest and XGBoost regressor and classifier models.
4. **Evaluate Performance**: Compute R2, RMSE for regression targets, and Accuracy/ROC-AUC for classification targets.
5. **Export Best Models**: Export a serialized pipeline as `bottleneck_model.pkl` containing the best regression models.

In [1]:
import os
import pickle
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, accuracy_score, classification_report, roc_auc_score
from xgboost import XGBRegressor, XGBClassifier

print("Libraries loaded successfully.")

Libraries loaded successfully.


## 1. Load Data & Prepare Targets
We load the integrated manufacturing dataset and construct our targets:
1. `bottleneck_severity_index` (continuous target)
2. `production_delay` = `target_quantity - actual_quantity` (continuous target)
3. `congestion_risk` = `1` if `utilization_rate > 90` else `0` (binary classification target)

In [2]:
PROCESSED_DIR = os.path.join("..", "datasets", "processed")
df = pd.read_csv(os.path.join(PROCESSED_DIR, "engineered_mfg_bottleneck.csv"))

# Calculate production delay
df['production_delay'] = df['target_quantity'] - df['actual_quantity']
# Define congestion risk binary target
df['congestion_risk'] = (df['utilization_rate'] > 90.0).astype(int)

# Feature columns
feature_cols = [
    'machine_id', 'vibration_mm_s', 'temperature_c', 'pressure_bar',
    'noise_level_db', 'sound_frequency_hz', 'sound_amplitude',
    'defect_count', 'energy_draw_kw'
]

X = df[feature_cols]
y_bottleneck = df['bottleneck_severity_index']
y_delay = df['production_delay']
y_congestion = df['congestion_risk']

# Train-test splits for each target
X_train, X_test, y_train_b, y_test_b = train_test_split(X, y_bottleneck, test_size=0.2, random_state=42)
_, _, y_train_d, y_test_d = train_test_split(X, y_delay, test_size=0.2, random_state=42)
_, _, y_train_c, y_test_c = train_test_split(X, y_congestion, test_size=0.2, random_state=42, stratify=y_congestion)

print("Train-test splits completed successfully.")

Train-test splits completed successfully.


## 2. Define Preprocessing Pipeline
We scale continuous variables and one-hot encode `machine_id`.

In [3]:
num_cols = ['vibration_mm_s', 'temperature_c', 'pressure_bar', 'noise_level_db', 'sound_frequency_hz', 'sound_amplitude', 'defect_count', 'energy_draw_kw']
cat_cols = ['machine_id']

preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(drop='first'), cat_cols)
])

print("Preprocessor pipeline ready.")

Preprocessor pipeline ready.


## 3. Train Bottleneck Severity Regressors
We train Random Forest and XGBoost regressors to predict the continuous Bottleneck Severity Index.

In [4]:
reg_models = {
    "Random Forest Regressor": RandomForestRegressor(n_estimators=100, random_state=42),
    "XGBoost Regressor": XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=5, random_state=42)
}

reg_results = {}

for name, model in reg_models.items():
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('regressor', model)
    ])
    
    pipeline.fit(X_train, y_train_b)
    y_pred = pipeline.predict(X_test)
    
    rmse = np.sqrt(mean_squared_error(y_test_b, y_pred))
    mae = mean_absolute_error(y_test_b, y_pred)
    r2 = r2_score(y_test_b, y_pred)
    
    reg_results[name] = {
        "RMSE": rmse,
        "MAE": mae,
        "R2-Score": r2,
        "Pipeline": pipeline
    }
    
    print(f"=== {name} Bottleneck Severity evaluation ===")
    print(f"  RMSE: {rmse:.4f}")
    print(f"  MAE:  {mae:.4f}")
    print(f"  R2:   {r2:.4f}")

=== Random Forest Regressor Bottleneck Severity evaluation ===
  RMSE: 6.4100
  MAE:  4.9432
  R2:   0.4063


=== XGBoost Regressor Bottleneck Severity evaluation ===
  RMSE: 6.4093
  MAE:  4.9550
  R2:   0.4064


## 4. Train Production Delay Regressors
We fit models to predict `production_delay`.

In [5]:
delay_results = {}

for name, model in reg_models.items():
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('regressor', model)
    ])
    
    pipeline.fit(X_train, y_train_d)
    y_pred = pipeline.predict(X_test)
    
    rmse = np.sqrt(mean_squared_error(y_test_d, y_pred))
    r2 = r2_score(y_test_d, y_pred)
    
    delay_results[name] = {
        "RMSE": rmse,
        "R2-Score": r2,
        "Pipeline": pipeline
    }
    print(f"{name} Production Delay R2: {r2:.4f}")

Random Forest Regressor Production Delay R2: -0.0034


XGBoost Regressor Production Delay R2: -0.0170


## 5. Congestion Risk Classifiers
We train XGBoost and Random Forest classifiers to flag high Congestion Risk.

In [6]:
clf_models = {
    "Random Forest Classifier": RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42),
    "XGBoost Classifier": XGBClassifier(n_estimators=100, learning_rate=0.1, random_state=42, eval_metric='logloss')
}

clf_results = {}

for name, model in clf_models.items():
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', model)
    ])
    
    pipeline.fit(X_train, y_train_c)
    y_pred = pipeline.predict(X_test)
    y_proba = pipeline.predict_proba(X_test)[:, 1]
    
    acc = accuracy_score(y_test_c, y_pred)
    roc = roc_auc_score(y_test_c, y_proba)
    
    clf_results[name] = {
        "Accuracy": acc,
        "ROC-AUC": roc,
        "Pipeline": pipeline
    }
    print(f"=== {name} Congestion Risk Evaluation ===")
    print(classification_report(y_test_c, y_pred))

=== Random Forest Classifier Congestion Risk Evaluation ===
              precision    recall  f1-score   support

           0       0.97      1.00      0.98      1252
           1       0.00      0.00      0.00        45

    accuracy                           0.97      1297
   macro avg       0.48      0.50      0.49      1297
weighted avg       0.93      0.97      0.95      1297



C:\Users\anime\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\anime\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\anime\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(ave

=== XGBoost Classifier Congestion Risk Evaluation ===
              precision    recall  f1-score   support

           0       0.97      1.00      0.98      1252
           1       0.00      0.00      0.00        45

    accuracy                           0.96      1297
   macro avg       0.48      0.50      0.49      1297
weighted avg       0.93      0.96      0.95      1297



## 6. Export Serialized Models
We group the best performing models into a unified python wrapper dict and export it as `bottleneck_model.pkl`.

In [7]:
# Find best regressor for Bottleneck Score
best_reg_name = max(reg_results, key=lambda k: reg_results[k]['R2-Score'])
best_reg_pipeline = reg_results[best_reg_name]['Pipeline']

# Find best regressor for Delay
best_delay_name = max(delay_results, key=lambda k: delay_results[k]['R2-Score'])
best_delay_pipeline = delay_results[best_delay_name]['Pipeline']

# Find best classifier for Congestion
best_clf_name = max(clf_results, key=lambda k: clf_results[k]['ROC-AUC'])
best_clf_pipeline = clf_results[best_clf_name]['Pipeline']

print(f"Selected Bottleneck Regressor: {best_reg_name}")
print(f"Selected Delay Regressor:      {best_delay_name}")
print(f"Selected Congestion Classifier: {best_clf_name}")

bottleneck_package = {
    'bottleneck_regressor': best_reg_pipeline,
    'delay_regressor': best_delay_pipeline,
    'congestion_classifier': best_clf_pipeline
}

model_path = "bottleneck_model.pkl"
with open(model_path, 'wb') as f:
    pickle.dump(bottleneck_package, f)

print(f"Serialized bottleneck package exported to: {os.path.abspath(model_path)}")

Selected Bottleneck Regressor: XGBoost Regressor
Selected Delay Regressor:      Random Forest Regressor
Selected Congestion Classifier: XGBoost Classifier
Serialized bottleneck package exported to: D:\PROJECTS\NexTwinAI\notebooks\bottleneck_model.pkl
